# Healthcare Appointment Access and Attendance Analysis
## Exploratory Data Analysis (EDA) Notebook

Welcome to the Exploratory Data Analysis (EDA) notebook for the **Healthcare Appointment Access and Attendance Analysis** project.
This notebook guides you through an end-to-end data analysis workflow using Python, Pandas, SQLite, Matplotlib, and Seaborn.

## 1. Project Introduction

Healthcare appointment non-attendance (often referred to as "no-shows") is a major logistical and financial challenge for medical clinics, hospitals, and public health systems worldwide. When patients miss appointments without prior cancellation:
- Clinical staff and medical resources remain underutilized.
- Waiting lists for other patients seeking timely care are artificially lengthened.
- Continuity of patient care is disrupted, which can worsen patient health outcomes for chronic conditions.

In this project, we explore historical appointment attendance data to understand why patients miss appointments and discover actionable opportunities to increase attendance rates.

## 2. Business Problem

Healthcare administrators need data-driven answers to several key business questions:
1. **Baseline Volume & Losses:** What proportion of booked appointments result in no-shows?
2. **Demographics:** Are specific age groups or genders more prone to missing appointments?
3. **Communication Effectiveness:** Do automated SMS reminders demonstrably improve attendance rates?
4. **Scheduling Dynamics:** How does the lead time (number of days between booking and the appointment) affect attendance?
5. **Operational Scheduling:** Are certain days of the week or clinic locations associated with higher no-show rates?

## 3. Dataset Description

We use the **Medical Appointment No Shows** dataset from Kaggle (~110,527 records from public clinics in Brazil).

Key variables:
- `PatientId` (`patient_id`): Identification of a patient.
- `AppointmentID` (`appointment_id`): Identification of each appointment.
- `Gender` (`gender`): Gender of the patient (Male `M` or Female `F`).
- `ScheduledDay` (`scheduled_day`): The day someone called or registered the appointment.
- `AppointmentDay` (`appointment_day`): The day of the actual appointment.
- `Age` (`age`): How old the patient is.
- `Neighbourhood` (`neighbourhood`): Where the clinic is located.
- `Scholarship` (`scholarship`): 1 or 0 (indicates whether patient receives welfare assistance under *Bolsa Família*).
- `Hipertension` (`hipertension`): 1 or 0 (diagnosed hypertension).
- `Diabetes` (`diabetes`): 1 or 0 (diagnosed diabetes).
- `Alcoholism` (`alcoholism`): 1 or 0 (alcoholism condition).
- `Handcap` (`handcap`): 0 to 4 (handicap severity/count).
- `SMS_received` (`sms_received`): 1 or 0 (1 or more text messages sent to the patient).
- `No-show` (`no_show`): 'No' (Patient attended) or 'Yes' (Patient did not show up).

## 4. Importing Libraries

We import our core data manipulation, database connection, and visualization libraries. We also configure visualization aesthetics using Seaborn and Matplotlib.

In [ ]:
import os
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

print("Libraries successfully imported!")

## 5. Connecting to SQLite Database & Loading the Dataset

### How Python Connects to SQLite with `sqlite3` and `pd.read_sql_query`

In professional data analytics, structured data is typically stored in a relational database rather than flat files. Python includes a built-in module named `sqlite3` to interface directly with SQLite databases without installing external database drivers.

**Key steps in the database connection process:**
1. **Determine Relative Path:** Using `pathlib.Path("../healthcare_analytics.db")`, we target the database file stored at the workspace root relative to this notebook in `notebooks/`.
2. **Open Connection (`sqlite3.connect`):** Connects to the SQLite file. If the file exists, it opens a session.
3. **Verify Table and Records:** We query the SQLite catalog (`sqlite_master`) to ensure the `appointments` table exists and run `SELECT COUNT(*)` to ensure it contains imported records.
4. **Query directly into DataFrame (`pd.read_sql_query`):** Pandas converts the SQL query results directly into a structured DataFrame in memory.
5. **Fallback to Raw CSV:** If the database table is not yet populated, the script falls back to reading `../data/raw/KaggleV2-May-2016.csv` so you can still run exploratory analysis.

In [ ]:
# Relative paths from the notebooks/ directory
db_path = Path("../healthcare_analytics.db")
raw_csv_path = Path("../data/raw/KaggleV2-May-2016.csv")

df = None

# Step 1: Attempt to load from the SQLite database
if db_path.exists():
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Check if 'appointments' table exists
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='appointments';")
        table_exists = cursor.fetchone()
        
        if table_exists:
            cursor.execute("SELECT COUNT(*) FROM appointments;")
            record_count = cursor.fetchone()[0]
            
            if record_count > 0:
                query = "SELECT * FROM appointments;"
                df = pd.read_sql_query(query, conn)
                print(f"[SUCCESS] Loaded {len(df):,} records directly from SQLite database '{db_path.name}' via pd.read_sql_query().")
            else:
                print(f"[INFO] Database table 'appointments' exists but is currently empty (0 records).")
        else:
            print(f"[INFO] Table 'appointments' does not exist yet in '{db_path.name}'.")
            
        conn.close()
    except sqlite3.Error as e:
        print(f"[WARNING] SQLite query error: {e}")

# Step 2: Fallback to raw CSV if database table is not yet loaded
if df is None:
    if raw_csv_path.exists():
        print(f"[INFO] Loading dataset from raw CSV file: {raw_csv_path}...")
        df = pd.read_csv(raw_csv_path)
        print(f"[SUCCESS] Loaded {len(df):,} rows and {len(df.columns)} columns from CSV.")
    else:
        print("[NOTICE] Neither database table records nor raw CSV file found.")
        print(f" - Place 'KaggleV2-May-2016.csv' into: {raw_csv_path.resolve()}")
        print(f" - Or run 'python scripts/import_data.py' to import records into {db_path.name}.")

## 6. Initial Data Inspection

Let's inspect the first few rows, columns, data types, and non-null counts.

In [ ]:
if df is not None:
    print("--- Column Data Types & Info ---")
    print(df.info())
    print("\n--- First 5 Rows (Head) ---")
    display(df.head())
else:
    print("Dataset not loaded.")

## 7. Data Quality Checks

We verify data integrity:
- Standardize column names to `snake_case` (if loaded from CSV).
- Convert date strings to Pandas `datetime` objects.
- Create a numeric binary indicator `is_no_show` (1 = Missed, 0 = Attended).

In [ ]:
if df is not None:
    # Standardize column names to snake_case if original CSV headers are present
    rename_dict = {
        "PatientId": "patient_id",
        "AppointmentID": "appointment_id",
        "Gender": "gender",
        "ScheduledDay": "scheduled_day",
        "AppointmentDay": "appointment_day",
        "Age": "age",
        "Neighbourhood": "neighbourhood",
        "Scholarship": "scholarship",
        "Hipertension": "hipertension",
        "Diabetes": "diabetes",
        "Alcoholism": "alcoholism",
        "Handcap": "handcap",
        "SMS_received": "sms_received",
        "No-show": "no_show"
    }
    df = df.rename(columns=rename_dict)
    
    # Parse dates to datetime objects
    df["scheduled_day"] = pd.to_datetime(df["scheduled_day"])
    df["appointment_day"] = pd.to_datetime(df["appointment_day"])
    
    # Create binary indicator for no_show (1 = Missed, 0 = Attended)
    df["is_no_show"] = (df["no_show"] == "Yes").astype(int)
    
    print("Data quality validation complete: Columns normalized and dates converted.")
    print(df.dtypes)

## 8. Missing-Value Analysis

We inspect whether any attributes contain `NULL` or missing (`NaN`) values.

In [ ]:
if df is not None:
    missing_summary = df.isnull().sum().to_frame(name="missing_count")
    missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df)) * 100
    print("--- Missing Values Summary ---")
    display(missing_summary)
else:
    print("Dataset not loaded.")

## 9. Duplicate Analysis

We evaluate duplicate records at both the appointment level (`appointment_id`) and the entire record level.

In [ ]:
if df is not None:
    duplicate_appointments = df.duplicated(subset=["appointment_id"]).sum()
    full_duplicate_rows = df.duplicated().sum()
    unique_patients = df["patient_id"].nunique()
    
    print(f"Duplicate Appointment IDs: {duplicate_appointments}")
    print(f"Full Duplicate Rows: {full_duplicate_rows}")
    print(f"Unique Patients: {unique_patients:,} across {len(df):,} total appointments.")
else:
    print("Dataset not loaded.")

## 10. Age Distribution

We investigate patient age distribution across the 110,526 valid appointments. A separate in-memory subset `valid_age_df` is created (`df['age'] >= 0`) to safeguard data hygiene without altering the primary database.

In [ ]:
if df is not None:
    # Filter out anomalous negative ages into a separate in-memory DataFrame
    valid_age_df = df[df["age"] >= 0].copy()
    
    # Summary statistics
    print("--- Age Summary Statistics (Cleaned in-memory subset) ---")
    age_stats = valid_age_df["age"].describe()
    display(age_stats.to_frame(name="Age Statistics"))
    
    # Plotting
    plt.figure(figsize=(10, 5), dpi=300)
    sns.histplot(valid_age_df["age"], bins=30, kde=True, color="#2b5c8f", edgecolor="white", alpha=0.7)
    plt.title("Patient Age Distribution (Excluding Negative Age Anomaly)", fontsize=13, fontweight="bold", pad=12)
    plt.xlabel("Patient Age (Years)", fontsize=11)
    plt.ylabel("Number of Appointments", fontsize=11)
    
    # Add vertical mean and median lines
    mean_age = valid_age_df["age"].mean()
    median_age = valid_age_df["age"].median()
    plt.axvline(mean_age, color="#d9534f", linestyle="--", linewidth=2, label=f"Mean Age: {mean_age:.1f}")
    plt.axvline(median_age, color="#5cb85c", linestyle="-", linewidth=2, label=f"Median Age: {median_age:.1f}")
    plt.legend(frameon=True, facecolor="white", loc="upper right")
    
    # Save high-resolution visual
    save_path = Path("../visuals/02_age_distribution.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print(f"[SUCCESS] Chart saved to: {save_path}")
    plt.show()
else:
    print("Dataset not loaded.")


### Interpretation: Patient Age Distribution
- **Mean Age:** 37.1 years; **Median Age:** 37.0 years.
- **Bimodal Characteristics:** Peaks occur around infancy (`Age = 0`, 3,539 infants) and between ages 45–65.
- **Data Hygiene:** The single clerical anomaly (`Age = -1`) was excluded from this visual without modifying the database.

---

## 11. Appointment Attendance Distribution

We examine the portfolio baseline attendance outcome split: what proportion of booked appointments were attended vs. missed?

In [ ]:
if df is not None:
    # Summary Table
    outcome_counts = df["no_show"].value_counts()
    outcome_pct = (df["no_show"].value_counts(normalize=True) * 100).round(2)
    
    summary_outcome = pd.DataFrame({
        "Outcome Status": ["Attended (No)", "Missed (Yes)"],
        "Appointments Count": [outcome_counts.get("No", 0), outcome_counts.get("Yes", 0)],
        "Percentage (%)": [outcome_pct.get("No", 0), outcome_pct.get("Yes", 0)]
    })
    
    print("--- Appointment Outcome Summary Table ---")
    display(summary_outcome)
    
    # Visualization
    plt.figure(figsize=(8, 5), dpi=300)
    colors = ["#2e7d32", "#c62828"]  # Professional green (attended) and red (missed)
    ax = sns.barplot(data=summary_outcome, x="Outcome Status", y="Appointments Count", palette=colors)
    plt.title("Overall Appointment Attendance vs. No-Show Distribution", fontsize=13, fontweight="bold", pad=12)
    plt.xlabel("Appointment Outcome", fontsize=11)
    plt.ylabel("Number of Appointments", fontsize=11)
    plt.ylim(0, max(summary_outcome["Appointments Count"]) * 1.15)
    
    # Direct data labels on top of bars (both Count and Percentage)
    for i, row in summary_outcome.iterrows():
        count_val = row["Appointments Count"]
        pct_val = row["Percentage (%)"]
        ax.text(i, count_val + 2000, f"{count_val:,}\n({pct_val}%)", ha="center", va="bottom", fontsize=10, fontweight="bold")
    
    # Save visual
    save_path = Path("../visuals/01_outcome_distribution.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print(f"[SUCCESS] Chart saved to: {save_path}")
    plt.show()
else:
    print("Dataset not loaded.")


### Interpretation: Baseline Attendance Distribution
- **Attended:** 88,208 appointments (**79.81%**).
- **Missed (No-shows):** 22,319 appointments (**20.19%**).
- **Takeaway:** Approximately 1 out of every 5 scheduled consultations goes unfulfilled, establishing the baseline benchmark (20.19%) for all subsequent comparative segmentations.

---

## 12. No-Show Rate by Age Group

We segment patients into distinct developmental stages: Children (0–12), Youth (13–24), Young Adults (25–44), Middle-Aged Adults (45–64), and Seniors (65+).

In [ ]:
if df is not None:
    # Define exact SQL-aligned age categorization function
    def assign_sql_age_group(age):
        if age < 0:
            return "Invalid (<0)"
        elif 0 <= age <= 12:
            return "0-12 (Children)"
        elif 13 <= age <= 24:
            return "13-24 (Youth)"
        elif 25 <= age <= 44:
            return "25-44 (Young Adults)"
        elif 45 <= age <= 64:
            return "45-64 (Middle-Aged Adults)"
        else:
            return "65+ (Seniors)"
    
    # Apply categorization across full dataset
    df["age_group"] = df["age"].apply(assign_sql_age_group)
    
    # Ordered categories
    group_order = ["0-12 (Children)", "13-24 (Youth)", "25-44 (Young Adults)", "45-64 (Middle-Aged Adults)", "65+ (Seniors)"]
    
    # Summary Table across valid age records (age >= 0)
    valid_age_analysis = df[df["age"] >= 0].groupby("age_group", observed=False).agg(
        total_appointments=("appointment_id", "count"),
        attended_appointments=("is_no_show", lambda x: (x == 0).sum()),
        missed_appointments=("is_no_show", "sum"),
        no_show_rate_pct=("is_no_show", lambda x: np.round(x.mean() * 100, 2))
    ).reindex(group_order).reset_index()
    
    print("--- SQL-Aligned No-Show Rate by Patient Age Group Summary Table ---")
    display(valid_age_analysis)
    
    # Verification assertions
    total_valid = valid_age_analysis["total_appointments"].sum()
    total_attended = valid_age_analysis["attended_appointments"].sum()
    total_missed = valid_age_analysis["missed_appointments"].sum()
    print(f"Valid Age Records Reconciliation -> Total: {total_valid:,} (Expected: 110,526) | Attended: {total_attended:,} | Missed: {total_missed:,}")
    
    # Plotting
    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
    palette = sns.color_palette("Blues_r", len(valid_age_analysis))
    bars = ax.bar(valid_age_analysis["age_group"], valid_age_analysis["no_show_rate_pct"], color=palette, edgecolor="black", linewidth=0.7)
    
    ax.axhline(20.19, color="#d9534f", linestyle="--", linewidth=1.8, label="Overall Baseline: 20.19%")
    ax.set_title("No-Show Rate Across Patient Age Cohorts (SQL-Aligned)", fontsize=13, fontweight="bold", pad=12)
    ax.set_ylabel("No-Show Rate (%)", fontsize=11)
    ax.set_xlabel("Age Demographic Group", fontsize=11)
    ax.set_ylim(0, 35)
    ax.legend(frameon=True, facecolor="white", loc="upper right")
    
    for bar, row in zip(bars, valid_age_analysis.itertuples()):
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, yval + 0.7,
                f"{yval:.2f}%\n(n={row.total_appointments:,})",
                ha="center", va="bottom", fontsize=9, fontweight="bold")
    
    plt.tight_layout()
    save_path = Path("../visuals/03_noshow_by_age_group.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print(f"[SUCCESS] Chart saved to: {save_path}")
    plt.show()
else:
    print("Dataset not loaded.")


### Interpretation: No-Show Rate by Age Group (SQL-Reconciled)
- **Full Reconciliation with SQL:** The Python age grouping now uses the exact boundary logic from `sql/03_kpi_analysis.sql`:
  - `0-12 (Children)`: **21,036** appointments (20.47% no-show)
  - `13-24 (Youth)`: **16,231** appointments (25.62% no-show)
  - `25-44 (Young Adults)`: **28,865** appointments (22.38% no-show)
  - `45-64 (Middle-Aged Adults)`: **29,992** appointments (17.20% no-show)
  - `65+ (Seniors)`: **14,402** appointments (15.50% no-show)
  - `Invalid (<0)`: **1** appointment (excluded from visualization)
- **Total Volume Checks:** Sums to exactly **110,527** across all 6 groups, and exactly **110,526** across the 5 valid groups.
- **Key Behavioral Pattern:** Absenteeism peaks among **Youth (13–24)** at **25.62%** and steadily declines as age increases, reaching its lowest rate among **Seniors (65+)** at **15.50%**.

---

## 13. No-Show Rate by Gender

We compare appointment volume and attendance behavior between female and male patients.

In [ ]:
if df is not None:
    # Summary Table
    gender_analysis = df.groupby("gender").agg(
        total_appointments=("appointment_id", "count"),
        attended_appointments=("is_no_show", lambda x: (x == 0).sum()),
        missed_appointments=("is_no_show", "sum"),
        no_show_rate_pct=("is_no_show", lambda x: np.round(x.mean() * 100, 2)),
        attendance_rate_pct=("is_no_show", lambda x: np.round((1 - x.mean()) * 100, 2))
    ).reset_index()
    gender_analysis["gender_name"] = gender_analysis["gender"].map({"F": "Female", "M": "Male"})
    
    print("--- No-Show Rate by Gender Summary Table ---")
    display(gender_analysis[["gender_name", "total_appointments", "attended_appointments", "missed_appointments", "no_show_rate_pct", "attendance_rate_pct"]])
    
    # Visualization: 2 Subplots (Volume Breakdown & No-Show Rate)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=300)
    palette = ["#8e44ad", "#2980b9"]
    
    # Left plot: Total volume
    sns.barplot(data=gender_analysis, x="gender_name", y="total_appointments", palette=palette, ax=ax1)
    ax1.set_title("Appointment Volume by Gender", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Gender", fontsize=11)
    ax1.set_ylabel("Total Appointments", fontsize=11)
    ax1.set_ylim(0, max(gender_analysis["total_appointments"]) * 1.15)
    for i, row in gender_analysis.iterrows():
        ax1.text(i, row["total_appointments"] + 1500, f"{row['total_appointments']:,}", ha="center", va="bottom", fontsize=10, fontweight="bold")
        
    # Right plot: No-show rate
    sns.barplot(data=gender_analysis, x="gender_name", y="no_show_rate_pct", palette=palette, ax=ax2)
    ax2.set_title("No-Show Rate (%) by Gender", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Gender", fontsize=11)
    ax2.set_ylabel("No-Show Rate (%)", fontsize=11)
    ax2.set_ylim(0, 30)
    ax2.axhline(20.19, color="#d9534f", linestyle="--", linewidth=1.5, label="Baseline: 20.19%")
    ax2.legend(frameon=True, facecolor="white", loc="upper right")
    for i, row in gender_analysis.iterrows():
        ax2.text(i, row["no_show_rate_pct"] + 0.7, f"{row['no_show_rate_pct']}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
        
    plt.suptitle("Gender Comparison: Appointment Volume vs. No-Show Tendency", fontsize=13, fontweight="bold", y=1.03)
    plt.tight_layout()
    
    # Save visual
    save_path = Path("../visuals/04_noshow_by_gender.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print(f"[SUCCESS] Chart saved to: {save_path}")
    plt.show()
else:
    print("Dataset not loaded.")


### Interpretation: Gender Attendance Patterns
- **Volume Asymmetry:** Females account for **65.00%** (71,840) of all bookings, while males account for **35.00%** (38,687).
- **No-Show Consistency:** Despite females booking nearly double the appointments, their no-show rate (**20.31%**) is virtually identical to males (**19.97%**), a negligible difference of 0.34%.
- **Takeaway:** Gender alone is not a strong predictive indicator of whether a patient will miss an appointment.

---

## 14. No-Show Rate by SMS Reminders

We compare attendance outcomes between patients who received automated SMS notifications vs. those who did not.

In [ ]:
if df is not None:
    # Summary Table
    sms_summary = df.groupby("sms_received").agg(
        total_appointments=("appointment_id", "count"),
        attended_appointments=("is_no_show", lambda x: (x == 0).sum()),
        missed_appointments=("is_no_show", "sum"),
        no_show_rate_pct=("is_no_show", lambda x: np.round(x.mean() * 100, 2))
    ).reset_index()
    sms_summary["sms_status"] = sms_summary["sms_received"].map({0: "No SMS Received", 1: "SMS Received"})
    
    print("--- No-Show Rate by SMS Reminder Summary Table ---")
    display(sms_summary[["sms_status", "total_appointments", "attended_appointments", "missed_appointments", "no_show_rate_pct"]])
    
    # Visualization
    fig, ax = plt.subplots(figsize=(8, 5), dpi=300)
    colors = ["#457b9d", "#e63946"]
    bars = ax.bar(sms_summary["sms_status"], sms_summary["no_show_rate_pct"], color=colors, width=0.45, edgecolor="black", linewidth=0.8)
    ax.axhline(20.19, color="#6c757d", linestyle="--", linewidth=1.5, label="Overall Baseline: 20.19%")
    ax.set_title("No-Show Rate by SMS Reminder Status", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("SMS Notification Status", fontsize=11)
    ax.set_ylabel("No-Show Rate (%)", fontsize=11)
    ax.set_ylim(0, 35)
    ax.legend(frameon=True, facecolor="white", loc="upper left")
    
    for bar, row in zip(bars, sms_summary.itertuples()):
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, yval + 0.8,
                f"{yval:.2f}%\n({row.missed_appointments:,} / {row.total_appointments:,})",
                ha="center", va="bottom", fontsize=10, fontweight="bold")
    
    plt.tight_layout()
    save_path = Path("../visuals/05_noshow_by_sms.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print(f"[SUCCESS] Chart saved to: {save_path}")
    plt.show()
else:
    print("Dataset not loaded.")


### Interpretation: SMS Notification & Confounding Lead Time
- **Observed Rates:** Patients who received an SMS reminder had a **27.57%** no-show rate, compared to **16.70%** for those who received no SMS.
- **Confounding Variable (Not Causation):** This paradoxical pattern is driven by operational scheduling policies:
  - All **38,563 same-day appointments** were scheduled with 0 days notice and never received an SMS (`sms_received = 0`). Same-day visits naturally have a very low miss rate (4.65%).
  - When filtering exclusively for advance appointments (>0 days notice), the no-show rate is **27.57% with SMS** vs. **29.44% without SMS** (a modest 1.87% reduction).
- **Crucial Analytical Takeaway:** This reflects a statistical association, not proof that reminders cause higher absenteeism.

---

## 15. No-Show Rate by Scheduling Lead Time

Lead time is defined as the number of calendar days between booking registration (`scheduled_day`) and the consultation date (`appointment_day`). We exclude the 5 negative date anomalies from this operational analysis.

In [ ]:
if df is not None:
    # Calculate calendar lead time in days: appointment date minus scheduled date
    df["scheduled_date"] = pd.to_datetime(df["scheduled_day"]).dt.date
    df["appointment_date"] = pd.to_datetime(df["appointment_day"]).dt.date
    df["lead_time_days"] = (df["appointment_date"] - df["scheduled_date"]).apply(lambda d: d.days)
    
    # Filter out only the 5 negative date anomalies (lead_time_days >= 0)
    valid_lead_df = df[df["lead_time_days"] >= 0].copy()
    valid_lead_df["is_no_show"] = (valid_lead_df["no_show"] == "Yes").astype(int)
    
    # Group into defined operational buckets
    lead_bins = [-1, 0, 3, 7, 14, 30, 200]
    lead_labels = ["Same day (0 days)", "1–3 days", "4–7 days", "8–14 days", "15–30 days", "More than 30 days"]
    valid_lead_df["lead_group"] = pd.cut(valid_lead_df["lead_time_days"], bins=lead_bins, labels=lead_labels)
    
    # Aggregate summary
    lead_summary = valid_lead_df.groupby("lead_group", observed=False).agg(
        total_appointments=("appointment_id", "count"),
        attended_appointments=("is_no_show", lambda x: (x == 0).sum()),
        missed_appointments=("is_no_show", "sum"),
        no_show_rate_pct=("is_no_show", lambda x: np.round(x.mean() * 100, 2))
    ).reset_index()
    
    print("--- Reconciled No-Show Rate by Scheduling Lead Time Summary Table ---")
    display(lead_summary)
    
    # Verification checks
    total_records = lead_summary["total_appointments"].sum()
    total_attended = lead_summary["attended_appointments"].sum()
    total_missed = lead_summary["missed_appointments"].sum()
    print(f"Reconciled Totals -> Total: {total_records:,} | Attended: {total_attended:,} | Missed: {total_missed:,}")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
    palette = sns.color_palette("YlOrRd", len(lead_summary))
    bars = ax.bar(lead_summary["lead_group"], lead_summary["no_show_rate_pct"], color=palette, edgecolor="black", linewidth=0.7)
    ax.axhline(20.19, color="#2b2d42", linestyle="--", linewidth=1.5, label="Overall Baseline: 20.19%")
    ax.set_title("No-Show Rate by Scheduling Lead Time (Booking Horizon)", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Lead Time Group (Days Between Booking and Appointment)", fontsize=11)
    ax.set_ylabel("No-Show Rate (%)", fontsize=11)
    ax.set_ylim(0, 40)
    ax.legend(frameon=True, facecolor="white", loc="upper left")
    
    for bar, row in zip(bars, lead_summary.itertuples()):
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, yval + 0.8,
                f"{yval:.2f}%\n(n={row.total_appointments:,})",
                ha="center", va="bottom", fontsize=8.5, fontweight="bold")
    
    plt.xticks(rotation=15)
    plt.tight_layout()
    save_path = Path("../visuals/06_noshow_by_lead_time.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print(f"[SUCCESS] Chart saved to: {save_path}")
    plt.show()
else:
    print("Dataset not loaded.")


### Interpretation: The Lead-Time Booking Effect (Reconciled)
- **Reconciliation Verification:** Across all 6 groups, Total = **110,522**, Attended = **88,208**, Missed = **22,314** (exactly matching 110,527 minus the 5 negative date anomalies).
- **Same-Day Attendance:** When booked on the same day (`0 days`), the no-show rate is exceptionally low at **4.65%** (1,792 missed out of 38,563).
- **Steep Escalation:** Moving from same-day to just `1–3 days` causes the no-show rate to jump almost fivefold to **22.89%** (3,359 missed out of 14,675).
- **Plateau at Distant Horizons:** Miss rates climb steadily: `4–7 days` (25.20%), `8–14 days` (30.47%), `15–30 days` (32.59%), reaching **33.00%** for appointments booked >30 days in advance.
- **Operational Implication:** The longer the wait between booking and appointment date, the higher the risk of forgetting, conflict, or diminished perceived urgency.

---

## 16. Weekday-Wise Appointment Analysis

We evaluate appointment scheduling volume and attendance patterns across each day of the week (Monday through Sunday).

In [ ]:
if df is not None:
    weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"]
    df["weekday"] = df["appointment_day"].dt.day_name()
    
    weekday_summary = df.groupby("weekday").agg(
        total_appointments=("appointment_id", "count"),
        attended_appointments=("is_no_show", lambda x: (x == 0).sum()),
        missed_appointments=("is_no_show", "sum"),
        no_show_rate_pct=("is_no_show", lambda x: np.round(x.mean() * 100, 2))
    ).reindex(weekday_order).dropna().reset_index()
    
    print("--- Weekday-Wise Appointment Summary Table ---")
    display(weekday_summary)
    
    # Dual-axis visualization (Volume bars + No-Show rate line)
    fig, ax1 = plt.subplots(figsize=(10, 5.5), dpi=300)
    bars = ax1.bar(weekday_summary["weekday"], weekday_summary["total_appointments"], color="#4a90e2", alpha=0.8, edgecolor="black", linewidth=0.6, width=0.55, label="Booked Appointments (Volume)")
    ax1.set_ylabel("Total Appointments Booked", color="#1c4e80", fontsize=11, fontweight="bold")
    ax1.tick_params(axis="y", labelcolor="#1c4e80")
    ax1.set_ylim(0, max(weekday_summary["total_appointments"]) * 1.25)
    ax1.set_xlabel("Weekday of Appointment", fontsize=11)
    
    for bar in bars:
        yval = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width() / 2, yval + 400, f"{int(yval):,}", ha="center", va="bottom", fontsize=9, color="#1c4e80", fontweight="bold")
    
    ax2 = ax1.twinx()
    ax2.plot(weekday_summary["weekday"], weekday_summary["no_show_rate_pct"], color="#d90429", marker="o", linewidth=2.5, markersize=8, label="No-Show Rate (%)")
    ax2.set_ylabel("No-Show Rate (%)", color="#d90429", fontsize=11, fontweight="bold")
    ax2.tick_params(axis="y", labelcolor="#d90429")
    ax2.set_ylim(10, 30)
    ax2.grid(False)
    
    for x, y in zip(weekday_summary["weekday"], weekday_summary["no_show_rate_pct"]):
        ax2.text(x, y + 0.6, f"{y:.1f}%", ha="center", va="bottom", fontsize=9.5, color="#d90429", fontweight="bold")
    
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="upper right", frameon=True, facecolor="white")
    
    plt.title("Appointment Volume and No-Show Rate Across Weekdays", fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout()
    save_path = Path("../visuals/07_weekday_analysis.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print(f"[SUCCESS] Chart saved to: {save_path}")
    plt.show()
else:
    print("Dataset not loaded.")


### Interpretation: Weekday Demand & Stability
- **Volume Distribution:** Tuesdays (25,640) and Wednesdays (25,867) have the highest clinic traffic, closely followed by Mondays (22,715). Booking volume tapers on Thursdays (17,247) and Fridays (19,019).
- **Weekend Operations:** Only **39 total appointments** took place on Saturday, and 0 on Sunday, reflecting public clinic closure.
- **Remarkable No-Show Consistency:** Weekday no-show rates remain extraordinarily steady across Monday–Friday, fluctuating narrowly between **19.35% and 21.23%** (Friday having the mildest peak).
- **Insight:** The day of the week has virtually no bearing on attendance variance.

---

## 17. Neighbourhood-Wise Analysis

We evaluate appointment volume and no-show rates across clinic neighbourhoods, focusing on the top 10 highest-volume locations.

In [ ]:
if df is not None:
    neighbourhood_summary = df.groupby("neighbourhood").agg(
        total_appointments=("appointment_id", "count"),
        attended_appointments=("is_no_show", lambda x: (x == 0).sum()),
        missed_appointments=("is_no_show", "sum"),
        no_show_rate_pct=("is_no_show", lambda x: np.round(x.mean() * 100, 2))
    ).reset_index()
    
    # Select top 10 neighbourhoods by appointment volume
    top10_neighbourhoods = neighbourhood_summary.sort_values(by="total_appointments", ascending=False).head(10).copy()
    top10_sorted_by_rate = top10_neighbourhoods.sort_values(by="no_show_rate_pct", ascending=True)
    
    print("--- Top 10 Neighbourhoods by Appointment Volume Summary Table ---")
    display(top10_neighbourhoods[["neighbourhood", "total_appointments", "attended_appointments", "missed_appointments", "no_show_rate_pct"]])
    
    # Horizontal bar chart
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    colors = ["#2a9d8f" if r < 20.19 else "#e76f51" for r in top10_sorted_by_rate["no_show_rate_pct"]]
    bars = ax.barh(top10_sorted_by_rate["neighbourhood"], top10_sorted_by_rate["no_show_rate_pct"], color=colors, edgecolor="black", linewidth=0.7, height=0.6)
    ax.axhline(20.19, color="#264653", linestyle="--", linewidth=1.5, label="Overall Portfolio Baseline: 20.19%")
    
    ax.set_title("No-Show Rates in Top 10 Highest-Volume Clinic Neighbourhoods", fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("No-Show Rate (%)", fontsize=11)
    ax.set_ylabel("Clinic Neighbourhood", fontsize=11)
    ax.set_xlim(0, 32)
    ax.legend(frameon=True, facecolor="white", loc="lower right")
    
    for bar, row in zip(bars, top10_sorted_by_rate.itertuples()):
        xval = bar.get_width()
        ax.text(xval + 0.4, bar.get_y() + bar.get_height() / 2,
                f"{xval:.2f}% (vol={row.total_appointments:,})",
                ha="left", va="center", fontsize=9, fontweight="bold")
    
    plt.tight_layout()
    save_path = Path("../visuals/08_top10_neighbourhoods.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    print(f"[SUCCESS] Chart saved to: {save_path}")
    plt.show()
else:
    print("Dataset not loaded.")


### Interpretation: Neighbourhood Attendance Disparities
- **High Variance Among Top Volume Hubs:** Across the top 10 busiest clinic districts, no-show rates span from **15.84%** (Santa Martha) up to **26.27%** (Itararé) — a wide 10.43% spread.
- **Best Performing Hubs:** **Santa Martha** (15.84%), **Jardim da Penha** (16.28%), and **Tabuazeiro** (18.30%) perform significantly better than the portfolio baseline.
- **Struggling Hubs:** **Itararé** (26.27%) and **Jesus de Nazareth** (24.40%) experience higher absenteeism.
- **Analytical Care (Association vs. Causation):** Geographic location itself does not cause no-shows. These disparities likely reflect local public transit availability, walking topography, or socioeconomic differences that warrant local operational investigation.

---

## 18. Key Observations

- **Baseline Non-Attendance:** Portfolio baseline no-show rate is **20.19%** (22,319 missed out of 110,527).
- **Lead Time is the Dominant Predictor:** Same-day bookings have a tiny **4.65%** no-show rate, while appointments booked >14 days in advance exceed **32%**.
- **Age Curve:** Absenteeism declines monotonically with age: Youth (25.62%) $\rightarrow$ Young Adults (22.38%) $\rightarrow$ Middle-Aged (17.20%) $\rightarrow$ Seniors (15.50%).
- **Gender Parity:** Females (20.31%) and males (19.97%) exhibit virtually identical attendance tendencies.
- **SMS Confounder:** The apparent higher miss rate for SMS recipients (27.57% vs 16.70%) is explained by lead time: same-day appointments never received an SMS.
- **Weekday Invariance:** Monday through Friday exhibit virtually identical no-show rates (~19.3% to ~21.2%).

## 19. Business Recommendations

1. **Strategic Overbooking & Confirmation Queues:** For appointments scheduled >14 days in advance, introduce automated re-confirmation prompts 48 hours prior.
2. **Targeted Follow-Ups for High-Risk Demographics:** Youth and working-age cohorts may benefit from flexible scheduling options (e.g., evening clinics or weekend slots).
3. **Optimized Notification Cadence:** Experiment with multiple notification channels (WhatsApp, phone calls, SMS timing) rather than a single SMS text message.

## 20. Limitations and Future Scope

- **Limitations:**
  - The dataset lacks transportation accessibility metrics, socio-economic income data, weather indicators, or appointment specialty types (e.g., oncology vs. routine checkup).
  - Observed correlations should not be interpreted as definitive causal proof without controlled A/B testing.
- **Future Scope:**
  - Predictive classification modeling (e.g., Random Forest / Logistic Regression) to estimate individual patient no-show probabilities.
  - Dynamic scheduling optimization based on predicted attendance likelihood.